In [ ]:
# ── Make HiGlass views render as standalone HTML ──────────────────────────
# By default, modern higlass-python returns an anywidget for each view, which
# only works in a live Jupyter kernel. To make outputs survive `nbconvert`
# and the static notebook renderer in this docs site, monkey-patch
# `Viewconf._repr_mimebundle_` so each cell's last expression emits a full
# standalone `text/html` HiGlass page instead.
import json as _json, uuid as _uuid
import higlass as hg
from higlass.api import Viewconf

try:
    from higlass.api import gather_plugin_urls as _gather_plugin_urls
except ImportError:  # older higlass-python versions
    def _gather_plugin_urls(views):
        urls = {}
        for v in (views or []):
            for group in (
                getattr(v, 'tracks', None) and v.tracks.model_dump().values()
            ) or []:
                for t in group or []:
                    u = (t or {}).get('plugin_url') if isinstance(t, dict) else getattr(t, 'plugin_url', None)
                    if u: urls[u] = True
        return list(urls.keys())

_HTML_TEMPLATE = '''<!DOCTYPE html>
<html>
  <head>
    <link rel="stylesheet" href="https://esm.sh/higlass@{higlass_version}/dist/hglib.css">
    <script src="https://unpkg.com/requirejs-toggle"></script>
    {plugin_scripts}
    <script src="https://unpkg.com/requirejs-toggle"></script>
  </head>
  <body>
    <div id="{output_div}" style="width:100%;"></div>
  </body>
  <script type="module">
    import hglib from "https://esm.sh/higlass@{higlass_version}?deps=react@{react_version},react-dom@{react_version},pixi.js@{pixijs_version}";
    hglib.viewer(
      document.getElementById('{output_div}'),
      {viewconf},
    );
  </script>
</html>'''

def _viewconf_to_html(viewconf, plugin_urls=None,
                      higlass_version='1.12', react_version='17', pixijs_version='6'):
    plugin_urls = plugin_urls or []
    plugin_scripts = '\n    '.join(
        f'<script src="{u}"></script>' for u in plugin_urls
    )
    return _HTML_TEMPLATE.format(
        higlass_version=higlass_version,
        react_version=react_version,
        pixijs_version=pixijs_version,
        output_div=f'jupyter-hg-{_uuid.uuid4().hex}',
        plugin_scripts=plugin_scripts,
        viewconf=_json.dumps(viewconf),
    )

def _viewconf_html_mimebundle(self, include=None, exclude=None):
    plugin_urls = [] if self.views is None else _gather_plugin_urls(self.views)
    html = _viewconf_to_html(
        viewconf=self.model_dump(exclude_none=True),
        plugin_urls=plugin_urls,
    )
    return {"text/html": html}

Viewconf._repr_mimebundle_ = _viewconf_html_mimebundle


In [22]:
import higlass as hg
file = "/workspace/workdisk3/rudhra/Desktop/experiments/plots/quest/MMTBMP031124.mcool"
CHROM_SIZES = "/workspace/workdisk3/rudhra/database/main_chroms/hg38.chrom.sizes"
BEDPEDB = "/workspace/workdisk3/rudhra/Desktop/experiments/plots/final/SRR2312566.bedpedb"

In [24]:
from typing import Literal

# ── ArcsTrack Plugin Definition ───────────────────────────────────────────────
class ArcsTrack(hg.PluginTrack):
    type: Literal['1d-arcs'] = '1d-arcs'
    plugin_url: str = 'https://unpkg.com/higlass-arcs/dist/higlass-arcs.min.js'


In [25]:
genes = hg.remote(
    uid='OHJakQICQD6gTD7skx4EWA',
    server='https://higlass.io/api/v1',
    name='hg38 Genes'
).track('horizontal-gene-annotations', height=60)

In [26]:
cool_ts = hg.cooler(file)
chrom_ts = hg.chromsizes(CHROM_SIZES)
arc_ts = hg.bed2ddb(BEDPEDB)


In [27]:


_ref_track = arc_ts.track('2d-rectangle-domains')

print(f'✅ Tileset loaded')
print(f'   Server: {_ref_track.server}')
print(f'   UID:    {_ref_track.tilesetUid}')


✅ Tileset loaded
   Server: jupyter
   UID:    hg_717b2de82750


In [28]:

arcs = ArcsTrack(
    type='1d-arcs',
    tilesetUid=_ref_track.tilesetUid,
    server=_ref_track.server,
    height=150,
    plugin_url='https://unpkg.com/higlass-arcs/dist/higlass-arcs.min.js',
    options={
        'strokeColor': 'red',
        'strokeWidth': 2.0,
        'strokeOpacity': 0.8,
        'arcStyle': 'circle',
        'flip1D': 'yes',
        'name': 'ChromSpec Interactions'
    }
)

In [29]:
# heatmap    = cool_ts.track("heatmap")
linear_heatmap = cool_ts.track("linear-heatmap")
chrom_v    = chrom_ts.track("horizontal-chromosome-labels")

In [30]:
view = hg.view(
    (linear_heatmap,"center"),
    (chrom_v,"top"),(genes, 'top'),
    (arcs,"top")
)

view

In [ ]:
import higlass as hg
from typing import Literal

# ── ArcsTrack Plugin Definition ───────────────────────────────────────────────
class ArcsTrack(hg.PluginTrack):
    type: Literal['1d-arcs'] = '1d-arcs'
    plugin_url: str = 'https://unpkg.com/higlass-arcs/dist/higlass-arcs.min.js'

print('✅ ArcsTrack defined')

# ── Load beddb tileset ────────────────────────────────────────────────────────
# BEDDB_FILE = "/workspace/workdisk3/rudhra/Desktop/experiments/plots/paper/templates_with_nb_chromspec.beddb"
BEDDB_FILE = "/workspace/workdisk3/rudhra/Desktop/experiments/plots/final/SRR2312566.bedpedb"

tileset = hg.bed2ddb(BEDDB_FILE)
_ref_track = tileset.track('2d-rectangle-domains')

print(f'✅ Tileset loaded')
print(f'   Server: {_ref_track.server}')
print(f'   UID:    {_ref_track.tilesetUid}')

arcs = ArcsTrack(
    type='1d-arcs',
    tilesetUid=_ref_track.tilesetUid,
    server=_ref_track.server,
    height=300,
    plugin_url='https://unpkg.com/higlass-arcs/dist/higlass-arcs.min.js',
    options={
        'strokeColor': 'red',
        'strokeWidth': 2.0,
        'strokeOpacity': 0.8,
        'arcStyle': 'circle',
        'flip1D': 'yes',
        'name': 'ChromSpec Interactions'
    }
)

view = hg.view((arcs, 'center'), width=12, height=400)
view = view.domain(x=(1229163303, 1239164008))  # chr7, PET=94
view

✅ ArcsTrack defined
✅ Tileset loaded
   Server: jupyter
   UID:    hg_717ad7f36510
